In [ ]:
!pip -qq install transformers==4.46.3
import utils

In [ ]:
import getpass

def get_model_token(model_name, namespace="serving-deploy", password=None):
    import subprocess
    import base64
    
    # Prompt for password if not provided
    if password is None:
        password = getpass.getpass("Enter OpenShift password for 'developer' user: ")
    
    # First, log in as developer user
    login_cmd = [
        "oc", "login",
        "https://api.ocp4.example.com:6443",
        "-u", "developer",
        "-p", password,
        "--insecure-skip-tls-verify=true"
    ]
    login_result = subprocess.run(login_cmd, capture_output=True, text=True)
    
    if login_result.returncode != 0:
        print(f"Error logging in: {login_result.stderr}")
        return None
    
    # Now retrieve the token secret
    secret_name = f"default-name-{model_name}-sa"
    cmd = ["oc", "get", "secret", secret_name, "-n", namespace, "-o", "jsonpath={.data.token}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and result.stdout:
        return base64.b64decode(result.stdout).decode('utf-8')
    else:
        print(f"Error getting token: {result.stderr}")
        return None


# Retrieve authentication tokens
# The password will be prompted once and reused for both models
ocp_password = getpass.getpass("Enter OpenShift password for 'developer' user: ")
diabetes_auth_token = get_model_token("diabetes", password=ocp_password)
distilbert_auth_token = get_model_token("distilbert", password=ocp_password)

diabetes_url = "https://diabetes-serving-deploy.apps.ocp4.example.com/v2/models/diabetes/infer"
distilbert_url = "https://distilbert-serving-deploy.apps.ocp4.example.com/v2/models/distilbert/infer"

1. Validate that the diabetes model responds using the KServe V2 API.

In [ ]:
print("\nValidating diabetes model...\n")
diabetes_request = utils.prepare_diabetes_request()
print(f"Diabetes request:\n {diabetes_request}")
response = utils.send_inference_request(diabetes_url, diabetes_request, diabetes_auth_token)
output = response.json()["outputs"][0]
diabetes_probability = output["data"][1]
print(f"Probability of diabetes: {100 * diabetes_probability:.2f}%")


2. Validate that the DistilBERT model performs sentiment analysis using the KServe V2 API.

In [ ]:
prompt = "OpenShift AI is great!"

print(f"\nPerforming sentiment analysis on '{prompt}' ...\n")
tokens = utils.tokenize(prompt)
print(f"Tokens:\n {tokens}")

# Prepare request in KServe V2 API format
distilbert_request = utils.prepare_distilbert_request(tokens)
print(f"\nDistilBERT request (KServe V2 format):\n {distilbert_request}\n")

response = utils.send_inference_request(distilbert_url, distilbert_request, distilbert_auth_token)

# Parse the response
output = response.json()["outputs"][0]
logits = output["data"]

# DistilBERT outputs two scores: [negative_score, positive_score]
negative_score = logits[0]
positive_score = logits[1]

# Determine sentiment based on which score is higher
if positive_score > negative_score:
    sentiment = "POSITIVE"
    confidence = positive_score
else:
    sentiment = "NEGATIVE"
    confidence = negative_score

print(f"\nSentiment: {sentiment}")
print(f"Confidence score: {confidence:.2f}")
